# Step 3 — Emotion Analysis

Runs a pre-trained emotion classifier (`j-hartmann/emotion-english-distilroberta-base`) over every post in the classified dataset, adding a `dominant_emotion` column.

**Output:** `./pickles/classified_scripts_with_emotions.pkl`

## 0. Prerequisites

**Requires** (run prior notebooks first):
- `./pickles/classified_dating_scripts.pkl` — SetFit-classified posts (`40-expansion.ipynb`)
- `./pickles/topic_metadata_llm.pkl` — LLM topic labels (`20-llm_labeling.ipynb`)
- `./BERTopic/` — fitted BERTopic model (`10-topic_modelling.ipynb`)

In [1]:
import pickle
import pandas as pd
from bertopic import BERTopic

topic_model = BERTopic.load("./BERTopic", embedding_model="all-MiniLM-L6-v2")

df_posts = pd.read_pickle("./pickles/first-date_posts-all.pkl")
documents = df_posts["title"] + ". " + df_posts["selftext"].fillna("")
documents = documents.apply(lambda x: x.replace("\n", " "))
docs = documents.tolist()

print(f"✅ topic_model loaded — {len(topic_model.get_topic_info())} topics.")
print(f"✅ {len(docs)} documents reconstructed.")

/opt/homebrew/anaconda3/envs/RedditAnalysis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ topic_model loaded — 33 topics.
✅ 87155 documents reconstructed.


## 1. Load Classified Dataset

In [2]:
import pandas as pd

input_file = "./pickles/classified_dating_scripts.pkl"
output_file = "./pickles/classified_scripts_with_emotions.pkl"
metadata_file = "./pickles/topic_metadata_llm.pkl"

df = pd.read_pickle(input_file)
print(f"✅ Loaded {len(df)} rows.")

✅ Loaded 87155 rows.


## 2. Attach BERTopic Topic IDs

Joins BERTopic topic assignments back onto the classified dataset by matching document text.

In [3]:
print("Attaching BERTopic topic IDs via text matching...")
try:
    doc_info = topic_model.get_document_info(docs)

    df['match_text'] = (df["title"] + ". " + df["selftext"].fillna("")).astype(str).str.replace("\n", " ")
    text_to_topic = dict(zip(doc_info['Document'], doc_info['Topic']))
    df['Topic'] = df['match_text'].map(text_to_topic)
    df = df.drop(columns=['match_text'])

    unmapped = df['Topic'].isna().sum()
    print(f"✅ Topic IDs attached. Unmapped rows: {unmapped}")
except Exception as e:
    print(f"⚠️ Error: {e}")

Attaching BERTopic topic IDs via text matching...
✅ Topic IDs attached. Unmapped rows: 0


## 3. Merge Topic Metadata

Adds `CustomLabel` and `Description` from the LLM-generated topic labels produced in `20-llm_labeling.ipynb`.

In [4]:
print("Merging topic metadata...")
try:
    metadata_df = pd.read_pickle(metadata_file)

    for col in ['CustomLabel', 'Description']:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = df.merge(metadata_df[['Topic', 'CustomLabel', 'Description']], on='Topic', how='left')
    df['CustomLabel'] = df['CustomLabel'].fillna("Outlier / No Topic")
    print("✅ Merged CustomLabel and Description.")
except Exception as e:
    print(f"⚠️ Warning: {e}")

Merging topic metadata...
✅ Merged CustomLabel and Description.


## 4. Emotion Classification

Runs `j-hartmann/emotion-english-distilroberta-base` over every post. Predicts one of: `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, `surprise`.

**To adjust:** Set `device=0` for GPU, `-1` for CPU. Change `BATCH_SIZE` based on available memory.

In [5]:
from transformers import pipeline
from tqdm.auto import tqdm

BATCH_SIZE = 128

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=False,
    device=-1  # 0 for GPU, -1 for CPU
)

texts = (df["title"] + ". " + df["selftext"].fillna("")).astype(str).tolist()
results = []

print(f"Classifying emotions for {len(texts)} posts...")
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i + BATCH_SIZE]
    preds = emotion_classifier(batch, truncation=True, max_length=512)
    results.extend([p['label'] for p in preds])

df['dominant_emotion'] = results
print("✅ Emotion classification complete.")

Device set to use cpu


Classifying emotions for 87155 posts...


100%|██████████| 681/681 [1:35:31<00:00,  8.42s/it]   

✅ Emotion classification complete.


## 5. Save & Preview

In [7]:
df.to_pickle(output_file)
print(f"✅ Saved to {output_file}")

preview_cols = [c for c in ['Topic', 'CustomLabel', 'predicted_script', 'dominant_emotion'] if c in df.columns]
print(df[preview_cols].head())

✅ Saved to ./pickles/classified_scripts_with_emotions.pkl
   Topic                                   CustomLabel  \
0      9           Early Sexual Expectations in Dating   
1      7      Dating & Gift-Giving Etiquette (Flowers)   
2      2  Modern Dating Frustrations & Self-Perception   
3      7      Dating & Gift-Giving Etiquette (Flowers)   
4      3                Early Dating Texting Etiquette   

                      predicted_script dominant_emotion  
0  The Anxious & Calculated First Date          neutral  
1  The Anxious & Calculated First Date          sadness  
2  The Anxious & Calculated First Date          sadness  
3  The Anxious & Calculated First Date          neutral  
4               Calculated Ambivalence         surprise  
